<a href="https://colab.research.google.com/github/ayushhh026/RuppeRisk/blob/main/notebooks/Model11_Optuna_XGBoost_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install optuna
# Mount Drive and import required libraries

from google.colab import drive
drive.mount('/content/drive')

import os
import gc
import json
import numpy as np
import pandas as pd

import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 12.0 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
# Define paths for Model09/10 artifacts and the Optuna study database
RESULTS_PATH = "/content/drive/MyDrive/RupeeRisk/"

# This is the parquet saved during Model10's run. Model10 decided to revert
# to the Model09 baseline (303 features) after the domain-feature experiment
# did not clear the keep threshold, so this file correctly contains the
# Model09 feature set with data.
MODEL09_DATA_PATH = RESULTS_PATH + "model10_recommended_features.parquet"

OPTUNA_DB_PATH = RESULTS_PATH + "optuna_model11.db"

os.makedirs(RESULTS_PATH, exist_ok=True)

print("Model09 dataset:", MODEL09_DATA_PATH)
print("Optuna database:", OPTUNA_DB_PATH)

Model09 dataset: /content/drive/MyDrive/RupeeRisk/model10_recommended_features.parquet
Optuna database: /content/drive/MyDrive/RupeeRisk/optuna_model11.db


In [ ]:
# Load the 303-feature dataset
model09_data = pd.read_parquet(MODEL09_DATA_PATH)

print("Model09 dataset shape:", model09_data.shape)
display(model09_data.head())

Model09 dataset shape: (307511, 304)


,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,...,CC_CC_DRAWING_LIMIT_RATIO_MAX,CC_CC_DRAWING_LIMIT_RATIO_MEAN,CC_CC_DRAWING_LIMIT_RATIO_SUM,CC_CC_MIN_PAYMENT_RATIO_MIN,CC_CC_MIN_PAYMENT_RATIO_MEAN,CC_CC_MIN_PAYMENT_RATIO_SUM,CC_CARD_COUNT,CC_LATE_RATE,CC_SEVERE_DPD_RATE,HAS_CREDIT_CARD_HISTORY
0,1,Cash loans,M,N,Y,0,202500.0,24700.5,351000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,0,Cash loans,F,N,N,0,270000.0,35698.5,1129500.0,Family,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,0,Revolving loans,M,Y,Y,0,67500.0,6750.0,135000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,0,Cash loans,F,N,Y,0,135000.0,29686.5,297000.0,Unaccompanied,...,0.0,0.0,0.0,NaN,NaN,0.0,1.0,0.0,0.0,1
4,0,Cash loans,M,N,Y,0,121500.0,21865.5,513000.0,Unaccompanied,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [ ]:
# Confirm TARGET is present and verify expected feature count
if "TARGET" not in model09_data.columns:
    raise ValueError("TARGET column is missing from Model09 dataset.")

X = model09_data.drop(columns=["TARGET"])
y = model09_data["TARGET"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of features:", X.shape[1])
print("Expected Model09 feature count: 303")

X shape: (307511, 303)
y shape: (307511,)
Number of features: 303
Expected Model09 feature count: 303


In [ ]:
# Locked Model09 benchmark to compare against
MODEL09_ROC_AUC = 0.7872
MODEL09_PR_AUC = 0.2887

print("Locked Model09 ROC-AUC:", MODEL09_ROC_AUC)
print("Locked Model09 PR-AUC:", MODEL09_PR_AUC)

Locked Model09 ROC-AUC: 0.7872
Locked Model09 PR-AUC: 0.2887


In [ ]:
# Same 80/20 stratified split used throughout the project
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))
print("\nValidation target distribution:")
print(y_valid.value_counts(normalize=True))

Training shape: (246008, 303)
Validation shape: (61503, 303)

Train target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Validation target distribution:
TARGET
0    0.919272
1    0.080728
Name: proportion, dtype: float64


In [ ]:
# Separate numeric vs categorical columns
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 287
Categorical features: 16


In [ ]:
# Reusable function to build a fresh preprocessor
def build_preprocessor():
    numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

In [ ]:
# Reusable function to build an XGBoost model from a params dict
def build_xgb(params):
    return XGBClassifier(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        max_depth=params["max_depth"],
        min_child_weight=params["min_child_weight"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        gamma=params["gamma"],
        reg_alpha=params["reg_alpha"],
        reg_lambda=params["reg_lambda"],
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        # Keep consistent with MODEL01 finding: scale_pos_weight stays off (1.0 = default/off)
        scale_pos_weight=1.0,
        random_state=42,
        n_jobs=-1
    )

In [ ]:
# Optuna objective: 3-fold Stratified CV inside training data only,
# optimizing mean ROC-AUC across folds
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 250, 700),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 20.0, log=True)
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    fold_roc_auc = []
    fold_pr_auc = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train, y_train), start=1):
        X_tr = X_train.iloc[train_idx]
        X_va = X_train.iloc[valid_idx]
        y_tr = y_train.iloc[train_idx]
        y_va = y_train.iloc[valid_idx]

        preprocessor = build_preprocessor()
        model = build_xgb(params)

        pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
        pipeline.fit(X_tr, y_tr)

        valid_proba = pipeline.predict_proba(X_va)[:, 1]

        fold_roc = roc_auc_score(y_va, valid_proba)
        fold_pr = average_precision_score(y_va, valid_proba)

        fold_roc_auc.append(fold_roc)
        fold_pr_auc.append(fold_pr)

        trial.report(np.mean(fold_roc_auc), step=fold)

        if trial.should_prune():
            raise optuna.TrialPruned()

        del pipeline, preprocessor, model
        gc.collect()

    mean_roc_auc = np.mean(fold_roc_auc)
    mean_pr_auc = np.mean(fold_pr_auc)

    trial.set_user_attr("mean_pr_auc", float(mean_pr_auc))
    trial.set_user_attr("std_roc_auc", float(np.std(fold_roc_auc)))

    return mean_roc_auc

In [ ]:
# Create (or resume) the Optuna study, persisted to SQLite for resumability
study = optuna.create_study(
    study_name="Model11_XGBoost_Tuning",
    storage="sqlite:///" + OPTUNA_DB_PATH,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)

print("Optuna study ready.")
print("Existing trials:", len(study.trials))

[I 2026-08-24 11:38:35,244] Using an existing study with name 'Model11_XGBoost_Tuning' instead of creating a new one.


Optuna study ready.
Existing trials: 21


In [ ]:
# Run up to N_TRIALS new trials (can be re-run later to add more, since the study is persisted)
N_TRIALS =  max(0, 25 - len(study.trials))

print(f"Running up to {N_TRIALS} new Optuna trials...")
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True, show_progress_bar=True)
print("Optuna optimization complete.")

Running up to 4 new Optuna trials...


  0%|          | 0/4 [00:00<?, ?it/s]

[I 2026-08-24 11:46:17,623] Trial 21 finished with value: 0.785333680665062 and parameters: {'n_estimators': 524, 'learning_rate': 0.03258253167670959, 'max_depth': 6, 'min_child_weight': 15, 'subsample': 0.8200027594246849, 'colsample_bytree': 0.8685275261098719, 'gamma': 2.913565267371898, 'reg_alpha': 0.009167785859319854, 'reg_lambda': 0.734794360472441}. Best is trial 21 with value: 0.785333680665062.
[I 2026-08-24 11:51:55,329] Trial 22 pruned. 
[I 2026-08-24 11:57:54,960] Trial 23 pruned. 
[I 2026-08-24 12:05:41,777] Trial 24 finished with value: 0.7847998311406842 and parameters: {'n_estimators': 617, 'learning_rate': 0.030405390306101336, 'max_depth': 5, 'min_child_weight': 18, 'subsample': 0.8177752708446108, 'colsample_bytree': 0.9269296510115621, 'gamma': 4.324017144974966, 'reg_alpha': 0.016746224396759168, 'reg_lambda': 1.0087303791330984}. Best is trial 21 with value: 0.785333680665062.
Optuna optimization complete.


In [ ]:
# Report the best trial found
print("Best trial number:", study.best_trial.number)
print("Best mean CV ROC-AUC:", f"{study.best_value:.6f}")
print("Best mean CV PR-AUC:", f"{study.best_trial.user_attrs.get('mean_pr_auc', np.nan):.6f}")

print("\nBest parameters:")
for key, value in study.best_params.items():
    print(f"{key}: {value}")

Best trial number: 21
Best mean CV ROC-AUC: 0.785334
Best mean CV PR-AUC: 0.276462

Best parameters:
n_estimators: 524
learning_rate: 0.03258253167670959
max_depth: 6
min_child_weight: 15
subsample: 0.8200027594246849
colsample_bytree: 0.8685275261098719
gamma: 2.913565267371898
reg_alpha: 0.009167785859319854
reg_lambda: 0.734794360472441


In [15]:
# Build a summary table of the best trial's CV performance vs the Model09 benchmark
best_params = study.best_params.copy()
best_cv_roc_auc = study.best_value
best_cv_pr_auc = study.best_trial.user_attrs.get("mean_pr_auc", np.nan)
best_cv_std_roc_auc = study.best_trial.user_attrs.get("std_roc_auc", np.nan)

best_trial_summary = pd.DataFrame({
    "Metric": [
        "Best CV ROC-AUC", "Best CV PR-AUC", "CV ROC-AUC Std",
        "Model09 Validation ROC-AUC", "Model09 Validation PR-AUC"
    ],
    "Value": [best_cv_roc_auc, best_cv_pr_auc, best_cv_std_roc_auc, MODEL09_ROC_AUC, MODEL09_PR_AUC]
})

display(best_trial_summary)

,Metric,Value
0,Best CV ROC-AUC,0.785334
1,Best CV PR-AUC,0.276462
2,CV ROC-AUC Std,0.002621
3,Model09 Validation ROC-AUC,0.787200
4,Model09 Validation PR-AUC,0.288700


In [16]:
# Review all completed and pruned trials
trials_df = study.trials_dataframe()

print("Completed trials:", (trials_df["state"] == "COMPLETE").sum())
print("Pruned trials:", (trials_df["state"] == "PRUNED").sum())

display(trials_df.sort_values("value", ascending=False).head(10))

Completed trials: 18
Pruned trials: 4


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,user_attrs_mean_pr_auc,user_attrs_std_roc_auc,state
21,21,0.785334,2026-08-24 11:38:35.631567,2026-08-24 11:46:17.590077,0 days 00:07:41.958510,0.868528,2.913565,0.032583,6,15,524,0.009168,0.734794,0.820003,0.276462,0.002621,COMPLETE
18,18,0.785154,2026-08-24 03:42:36.200569,2026-08-24 03:50:36.674893,0 days 00:08:00.474324,0.900506,4.385402,0.027780,6,16,579,0.008787,0.107232,0.818580,0.276112,0.002528,COMPLETE
17,17,0.785015,2026-08-24 03:35:21.279477,2026-08-24 03:42:35.941998,0 days 00:07:14.662521,0.913957,4.171509,0.032043,5,19,598,0.111250,4.290819,0.832246,0.276702,0.002635,COMPLETE
24,24,0.784800,2026-08-24 11:57:55.203897,2026-08-24 12:05:41.746797,0 days 00:07:46.542900,0.926930,4.324017,0.030405,5,18,617,0.016746,1.008730,0.817775,0.276087,0.002614,COMPLETE
5,5,0.784715,2026-08-24 02:29:53.945246,2026-08-24 02:38:02.943996,0 days 00:08:08.998750,0.989355,3.875664,0.023260,6,11,548,4.983044,11.455778,0.714699,0.276248,0.001957,COMPLETE
15,15,0.784391,2026-08-24 03:20:54.198670,2026-08-24 03:27:30.335875,0 days 00:06:36.137205,0.926476,3.690074,0.058298,5,20,699,0.089867,7.090738,0.987573,0.275456,0.002434,COMPLETE
16,16,0.784320,2026-08-24 03:27:30.516877,2026-08-24 03:35:21.094103,0 days 00:07:50.577226,0.923397,2.793993,0.033195,5,20,693,0.081255,5.074763,0.999536,0.275382,0.002522,COMPLETE
14,14,0.783832,2026-08-24 03:15:02.334763,2026-08-24 03:20:54.020223,0 days 00:05:51.685460,0.982070,3.646875,0.075754,5,1,649,0.081196,19.371890,0.996627,0.274854,0.002113,COMPLETE
12,12,0.783810,2026-08-24 03:08:30.949251,2026-08-24 03:11:51.706453,0 days 00:03:20.757202,0.967683,3.639003,0.100072,4,1,271,0.000134,15.868994,0.874031,0.275188,0.001800,COMPLETE
19,19,0.783759,2026-08-24 03:50:36.872416,2026-08-24 03:57:08.798523,0 days 00:06:31.926107,0.889376,4.280512,0.032655,4,16,610,0.004614,0.123596,0.827622,0.275070,0.002231,COMPLETE


In [17]:
# Save the full trial history for reference
trials_output_path = RESULTS_PATH + "model11_optuna_trials.csv"
trials_df.to_csv(trials_output_path, index=False)
print("Optuna trials saved to:", trials_output_path)

Optuna trials saved to: /content/drive/MyDrive/RupeeRisk/model11_optuna_trials.csv


In [18]:
# Save the best hyperparameters as JSON
best_params_path = RESULTS_PATH + "model11_best_xgb_params.json"
with open(best_params_path, "w") as f:
    json.dump(best_params, f, indent=4)

print("Best parameters saved to:", best_params_path)

Best parameters saved to: /content/drive/MyDrive/RupeeRisk/model11_best_xgb_params.json


In [19]:
# Release the trials dataframe now that it's saved
trials_df = None
gc.collect()

print("Optuna trial table released from memory.")
!free -h

Optuna trial table released from memory.
               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.9Gi       6.2Gi       2.0Mi       2.6Gi       8.5Gi
Swap:             0B          0B          0B


In [20]:
# Build a fresh pipeline using the best-found hyperparameters
best_preprocessor = build_preprocessor()
best_xgb = build_xgb(best_params)

best_pipeline = Pipeline(steps=[
    ("preprocessor", best_preprocessor),
    ("model", best_xgb)
])

In [21]:
# Train the best configuration on the complete training set
print("Training best Optuna model on full training set...")
best_pipeline.fit(X_train, y_train)
print("Final training complete.")

Training best Optuna model on full training set...
Final training complete.


In [22]:
# Generate predictions on the untouched validation set - first time it's used since Cell 6
valid_proba = best_pipeline.predict_proba(X_valid)[:, 1]
print("Validation predictions generated.")

Validation predictions generated.


In [23]:
# Compute the final ROC-AUC and PR-AUC on the untouched validation set
tuned_roc_auc = roc_auc_score(y_valid, valid_proba)
tuned_pr_auc = average_precision_score(y_valid, valid_proba)

print("============================================================")
print("MODEL11 OPTUNA FINAL VALIDATION")
print(f"ROC-AUC: {tuned_roc_auc:.4f}")
print(f"PR-AUC:  {tuned_pr_auc:.4f}")
print("============================================================")

MODEL11 OPTUNA FINAL VALIDATION
ROC-AUC: 0.7882
PR-AUC:  0.2905


In [24]:
# Measure improvement over the locked Model09 benchmark, and distance from the 0.80 target
roc_change = tuned_roc_auc - MODEL09_ROC_AUC
pr_change = tuned_pr_auc - MODEL09_PR_AUC

print("IMPROVEMENT OVER MODEL09")
print(f"ROC-AUC change: {roc_change:+.4f}")
print(f"PR-AUC change:  {pr_change:+.4f}")
print("\nDistance from 0.80 ROC-AUC target:", f"{0.8000 - tuned_roc_auc:+.4f}")

IMPROVEMENT OVER MODEL09
ROC-AUC change: +0.0010
PR-AUC change:  +0.0018

Distance from 0.80 ROC-AUC target: +0.0118


In [25]:
# Build a summary comparison table
optuna_result = pd.DataFrame({
    "Experiment": ["Model09 Baseline", "Model11 Optuna XGBoost"],
    "ROC-AUC": [MODEL09_ROC_AUC, tuned_roc_auc],
    "PR-AUC": [MODEL09_PR_AUC, tuned_pr_auc],
    "ROC-AUC Change": [0.0, roc_change],
    "PR-AUC Change": [0.0, pr_change],
    "Feature Count": [X_train.shape[1], X_train.shape[1]]
})

display(optuna_result.style.format({
    "ROC-AUC": "{:.4f}", "PR-AUC": "{:.4f}",
    "ROC-AUC Change": "{:+.4f}", "PR-AUC Change": "{:+.4f}"
}))

,Experiment,ROC-AUC,PR-AUC,ROC-AUC Change,PR-AUC Change,Feature Count
0,Model09 Baseline,0.7872,0.2887,+0.0000,+0.0000,303
1,Model11 Optuna XGBoost,0.7882,0.2905,+0.0010,+0.0018,303


In [26]:
# Save the final result comparison table
optuna_result_path = RESULTS_PATH + "model11_optuna_results.csv"
optuna_result.to_csv(optuna_result_path, index=False)
print("Optuna result saved to:", optuna_result_path)

Optuna result saved to: /content/drive/MyDrive/RupeeRisk/model11_optuna_results.csv


In [27]:
!pip install joblib
# Save the fully trained, tuned pipeline for deployment/SHAP use
import joblib

model_path = RESULTS_PATH + "rupeerisk_model11_optuna_xgb.joblib"
joblib.dump(best_pipeline, model_path)
print("Final tuned pipeline saved to:", model_path)

Final tuned pipeline saved to: /content/drive/MyDrive/RupeeRisk/rupeerisk_model11_optuna_xgb.joblib


In [28]:
# Install MLflow in this Colab session
!pip install mlflow -q
import mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.2/216.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.

In [29]:
# Point at the same tracking database used throughout the project
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/RupeeRisk/mlflow.db")
mlflow.set_experiment("RupeeRisk")

<Experiment: artifact_location='/content/mlruns/1', creation_time=1787393296537, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787393296537, lifecycle_stage='active', name='RupeeRisk', tags={}, trace_location=None, workspace='default'>

In [30]:
# Record this stage's parameters and metrics
with mlflow.start_run(run_name="XGBoost_Optuna_Model11"):
    mlflow.log_param("stage", "Model11 - Optuna XGBoost Tuning")
    mlflow.log_param("input_features", "Model09 recommended feature set")
    mlflow.log_param("feature_count", X_train.shape[1])
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("optimization_metric", "ROC-AUC")
    mlflow.log_param("scale_pos_weight", 1.0)
    mlflow.log_param("n_trials", N_TRIALS)

    for key, value in best_params.items():
        mlflow.log_param(key, value)

    mlflow.log_metric("best_cv_roc_auc", best_cv_roc_auc)
    if not np.isnan(best_cv_pr_auc):
        mlflow.log_metric("best_cv_pr_auc", best_cv_pr_auc)

    mlflow.log_metric("validation_roc_auc", tuned_roc_auc)
    mlflow.log_metric("validation_pr_auc", tuned_pr_auc)
    mlflow.log_metric("roc_auc_change_vs_Model09", roc_change)
    mlflow.log_metric("pr_auc_change_vs_Model09", pr_change)

print("Model11 logged to MLflow.")

Model11 logged to MLflow.


In [31]:
# Pull every run logged so far, sorted by validation ROC-AUC
runs = mlflow.search_runs(experiment_names=["RupeeRisk"])
display(
    runs[["tags.mlflow.runName", "metrics.validation_roc_auc", "metrics.validation_pr_auc",
          "metrics.roc_auc", "metrics.pr_auc"]]
    .sort_values(by="metrics.validation_roc_auc", ascending=False)
)

,tags.mlflow.runName,metrics.validation_roc_auc,metrics.validation_pr_auc,metrics.roc_auc,metrics.pr_auc
0,XGBoost_Optuna_Model11,0.788152,0.290469,NaN,NaN
1,XGBoost_Polynomial_Domain,NaN,NaN,0.786862,0.289147
2,XGBoost_Feature_Refinement,NaN,NaN,NaN,NaN
3,XGBoost_Credit_Card,NaN,NaN,0.787019,0.287171
4,XGBoost_Installments,NaN,NaN,0.785949,0.286362
5,XGBoost_POS_CASH,NaN,NaN,0.783313,0.278581
6,XGBoost_Bureau,NaN,NaN,0.777585,0.274161
7,XGBoost_Previous_Application,NaN,NaN,0.775428,0.265853
8,XGBoost_Application_Features,NaN,NaN,0.769403,0.262725
9,XGBoost_scale_pos_weight,NaN,NaN,0.760000,0.249300


In [32]:
# Print the overall summary of the Optuna tuning stage
print("""
MODEL11 OPTUNA XGBOOST TUNING COMPLETE

Model09 benchmark:     ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Best Optuna CV:        ROC-AUC = {:.4f}, PR-AUC = {:.4f}
Final untouched valid: ROC-AUC = {:.4f}, PR-AUC = {:.4f}

Validation improvement: ROC-AUC = {:+.4f}, PR-AUC = {:+.4f}
Distance from 0.80 ROC-AUC: {:+.4f}

Best parameters:
{}
""".format(
    MODEL09_ROC_AUC, MODEL09_PR_AUC,
    best_cv_roc_auc, best_cv_pr_auc,
    tuned_roc_auc, tuned_pr_auc,
    roc_change, pr_change,
    0.8000 - tuned_roc_auc,
    json.dumps(best_params, indent=2)
))


MODEL11 OPTUNA XGBOOST TUNING COMPLETE

Model09 benchmark:     ROC-AUC = 0.7872, PR-AUC = 0.2887
Best Optuna CV:        ROC-AUC = 0.7853, PR-AUC = 0.2765
Final untouched valid: ROC-AUC = 0.7882, PR-AUC = 0.2905

Validation improvement: ROC-AUC = +0.0010, PR-AUC = +0.0018
Distance from 0.80 ROC-AUC: +0.0118

Best parameters:
{
  "n_estimators": 524,
  "learning_rate": 0.03258253167670959,
  "max_depth": 6,
  "min_child_weight": 15,
  "subsample": 0.8200027594246849,
  "colsample_bytree": 0.8685275261098719,
  "gamma": 2.913565267371898,
  "reg_alpha": 0.009167785859319854,
  "reg_lambda": 0.734794360472441
}



In [33]:
# Final RAM check
gc.collect()
print("Final RAM status:")
!free -h

Final RAM status:
               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.6Gi       6.0Gi       2.0Mi       3.1Gi       8.7Gi
Swap:             0B          0B          0B
